In [2]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import random
import hashlib


In [3]:
np.random.seed(42)
random.seed(42)

In [4]:
def generate_customers(n_customers=10000):
    """Generate customer data"""
    customers = []
    for i in range(n_customers):
        # Customer segments
        segment = np.random.choice(['Retail', 'Premium', 'Business', 'Student'], 
                                  p=[0.4, 0.2, 0.3, 0.1])
        
        # Account age (in days)
        account_age = np.random.randint(30, 365*10)
        
        # Risk score (higher = higher risk)
        base_risk = np.random.normal(0.5, 0.15)
        if segment == 'Premium':
            base_risk -= 0.1
        elif segment == 'Business':
            base_risk += 0.05
        risk_score = np.clip(base_risk, 0, 1)
        
        customers.append({
            'customer_id': f'CUST_{i:06d}',
            'segment': segment,
            'account_age_days': account_age,
            'risk_score': risk_score,
            'email_verified': np.random.choice([True, False], p=[0.8, 0.2]),
            'phone_verified': np.random.choice([True, False], p=[0.7, 0.3]),
            'avg_monthly_transactions': np.random.randint(5, 100),
            'avg_transaction_amount': np.random.uniform(10, 5000),
            'device_count': np.random.randint(1, 5),
            'location': np.random.choice(['US', 'UK', 'IN', 'SG', 'AU'], 
                                        p=[0.4, 0.2, 0.2, 0.1, 0.1])
        })
    
    return pd.DataFrame(customers)

In [5]:
def generate_transactions(customers, n_transactions=100000, fraud_rate=0.02):
    """Generate transaction data with fraud patterns"""
    transactions = []
    
    # Fraud patterns
    fraud_patterns = [
        {'type': 'high_amount', 'amount': (50000, 200000), 'probability': 0.3},
        {'type': 'velocity', 'amount': (1000, 50000), 'probability': 0.25},
        {'type': 'unusual_hour', 'amount': (500, 50000), 'probability': 0.15},
        {'type': 'new_device', 'amount': (1000, 30000), 'probability': 0.15},
        {'type': 'unusual_location', 'amount': (1000, 50000), 'probability': 0.1},
        {'type': 'round_amount', 'amount': (100, 10000), 'probability': 0.05},
    ]
    
    # Generate transactions
    for i in range(n_transactions):
        customer = customers.iloc[np.random.randint(0, len(customers))]
        
        # Base transaction
        is_fraud = np.random.random() < fraud_rate
        
        # Transaction amount based on fraud status
        if is_fraud:
            pattern = np.random.choice(fraud_patterns, p=[p['probability'] for p in fraud_patterns])
            amount = np.random.uniform(pattern['amount'][0], pattern['amount'][1])
        else:
            amount = np.random.uniform(5, customer['avg_transaction_amount'] * 2)
        
        # Transaction time
        if is_fraud and np.random.random() < 0.3:
            hour = np.random.choice([0, 1, 2, 3, 22, 23])
        else:
            hour = np.random.randint(6, 22)
        
        # Transaction location
        if is_fraud and np.random.random() < 0.2:
            location = np.random.choice(['RU', 'CN', 'NG', 'BR'])
        else:
            location = customer['location']
        
        # Device
        is_new_device = is_fraud and np.random.random() < 0.3
        
        # Transaction
        tx = {
            'transaction_id': f'TX_{i:08d}',
            'customer_id': customer['customer_id'],
            'amount': round(amount, 2),
            'timestamp': datetime.now() - timedelta(days=np.random.randint(0, 30),
                                                   hours=np.random.randint(0, 24),
                                                   minutes=np.random.randint(0, 60)),
            'hour': hour,
            'day_of_week': np.random.randint(0, 7),
            'location': location,
            'merchant_category': np.random.choice(['Retail', 'E-commerce', 'Travel', 'Food', 'Entertainment', 'Services']),
            'is_new_device': is_new_device,
            'device_id': f'DEV_{np.random.randint(1, 5000):04d}',
            'ip_address': f'{np.random.randint(1,255)}.{np.random.randint(1,255)}.{np.random.randint(1,255)}.{np.random.randint(1,255)}',
            'is_fraud': int(is_fraud),
            'transaction_type': np.random.choice(['online', 'in-store', 'atm', 'wire']),
            'card_present': np.random.choice([True, False], p=[0.6, 0.4])
        }
        transactions.append(tx)
    
    df_transactions = pd.DataFrame(transactions)
    df_transactions['timestamp'] = pd.to_datetime(df_transactions['timestamp'])
    
    return df_transactions

In [6]:
def generate_fraud_cases(transactions):
    """Generate fraud case data"""
    fraud_txs = transactions[transactions['is_fraud'] == 1]
    
    fraud_cases = []
    for i, tx in fraud_txs.iterrows():
        fraud_cases.append({
            'case_id': f'FC_{i:06d}',
            'transaction_id': tx['transaction_id'],
            'customer_id': tx['customer_id'],
            'fraud_amount': tx['amount'],
            'detection_time': tx['timestamp'] + timedelta(minutes=np.random.randint(1, 60)),
            'fraud_type': np.random.choice(['High Amount', 'Velocity', 'Suspicious Pattern', 'Location Mismatch', 'Device Anomaly']),
            'status': np.random.choice(['Under Review', 'Confirmed', 'False Positive'], p=[0.3, 0.5, 0.2]),
            'investigator': f'INV_{np.random.randint(1, 20):03d}' if np.random.random() > 0.5 else None
        })
    
    return pd.DataFrame(fraud_cases)

def generate_alerts(transactions):
    """Generate alert data"""
    alerts = []
    
    # Generate alerts for suspicious transactions (including some non-fraud)
    suspicious_txs = transactions[
        (transactions['amount'] > transactions['amount'].quantile(0.95)) |
        (transactions['is_new_device'] == True) |
        (transactions['hour'].isin([0, 1, 2, 3, 22, 23]))
    ].sample(min(len(transactions) // 20, 5000))
    
    for i, tx in suspicious_txs.iterrows():
        alert_types = ['High Amount', 'Unusual Time', 'New Device', 'Suspicious Location']
        if tx['is_fraud'] == 1:
            alert_type = np.random.choice(alert_types, p=[0.35, 0.2, 0.25, 0.2])
        else:
            alert_type = np.random.choice(alert_types, p=[0.2, 0.2, 0.3, 0.3])
        
        alerts.append({
            'alert_id': f'ALT_{i:06d}',
            'transaction_id': tx['transaction_id'],
            'customer_id': tx['customer_id'],
            'alert_type': alert_type,
            'severity': np.random.choice(['Low', 'Medium', 'High', 'Critical'], 
                                       p=[0.2, 0.4, 0.3, 0.1]),
            'timestamp': tx['timestamp'],
            'is_resolved': np.random.choice([True, False], p=[0.6, 0.4])
        })
    
    return pd.DataFrame(alerts)


In [7]:
print("Generating customer data...")
customers_df = generate_customers(10000)

print("Generating transactions...")
transactions_df = generate_transactions(customers_df, 100000, 0.02)

print("Generating fraud cases...")
fraud_cases_df = generate_fraud_cases(transactions_df)

print("Generating alerts...")
alerts_df = generate_alerts(transactions_df)

Generating customer data...
Generating transactions...
Generating fraud cases...
Generating alerts...


In [8]:
print("Saving data...")
customers_df.to_csv(r'D:\Dataset\Internship Fraud Detection\customers.csv', index=False)
transactions_df.to_csv(r'D:\Dataset\Internship Fraud Detection\paysim_transactions.csv', index=False)
fraud_cases_df.to_csv(r'D:\Dataset\Internship Fraud Detection\fraud_cases.csv', index=False)
alerts_df.to_csv(r'D:\Dataset\Internship Fraud Detection\alerts.csv', index=False)

Saving data...


In [9]:
print("Data generation complete!")
print(f"Customers: {len(customers_df)}")
print(f"Transactions: {len(transactions_df)}")
print(f"Fraud Cases: {len(fraud_cases_df)}")
print(f"Alerts: {len(alerts_df)}")

Data generation complete!
Customers: 10000
Transactions: 100000
Fraud Cases: 1974
Alerts: 5000


In [17]:
transactions_df.head(5)

,transaction_id,customer_id,amount,timestamp,hour,day_of_week,location,merchant_category,is_new_device,device_id,ip_address,is_fraud,transaction_type,card_present
0,TX_00000000,CUST_003557,1697.75,2026-09-09 15:14:33.884401,6,4,IN,Food,False,DEV_0171,64.143.224.29,0,in-store,False
1,TX_00000001,CUST_006907,2618.68,2026-09-03 21:14:33.885119,21,0,IN,Entertainment,False,DEV_1792,120.118.241.96,0,wire,False
2,TX_00000002,CUST_003146,281.30,2026-09-06 07:23:33.885389,17,3,IN,E-commerce,False,DEV_0744,3.175.117.166,0,online,True
3,TX_00000003,CUST_002992,215.74,2026-08-18 07:15:33.885632,19,5,UK,E-commerce,False,DEV_3614,234.103.87.124,0,online,True
4,TX_00000004,CUST_006527,507.04,2026-09-01 03:39:33.885898,8,3,US,Services,False,DEV_3759,84.48.2.23,0,in-store,True


In [16]:
test = pd.read_csv(
    r"D:\Dataset\Internship Fraud Detection\test_transactions_100_v2.csv",
    encoding="latin1",
    sep=None,
    engine="python"
)

print(test.head())
print(test.shape)


ParserError: Expected 2 fields in line 5, saw 3

In [13]:
transactions_df.columns

Index(['transaction_id', 'customer_id', 'amount', 'timestamp', 'hour',
       'day_of_week', 'location', 'merchant_category', 'is_new_device',
       'device_id', 'ip_address', 'is_fraud', 'transaction_type',
       'card_present'],
      dtype='object')

In [14]:
transactions_df['location'].value_counts()

location
US    38867
UK    20446
IN    19629
AU    10633
SG    10040
CN       98
BR       97
RU       97
NG       93
Name: count, dtype: int64

In [15]:
transactions_df['is_fraud'].value_counts()

is_fraud
0    98045
1     1955
Name: count, dtype: int64

In [20]:
all = alerts_df.to_csv(r"C:\Users\user\Downloads\Financial fraud detection.zip\Financial fraud detection\alerts.csv", index=False)

In [22]:
alerts_df

,alert_id,transaction_id,customer_id,alert_type,severity,timestamp,is_resolved
0,ALT_079681,TX_00079681,CUST_007451,High Amount,High,2026-08-17 00:15:46.462118,True
1,ALT_045971,TX_00045971,CUST_001232,New Device,Critical,2026-08-24 02:20:38.266080,True
2,ALT_095835,TX_00095835,CUST_002090,Unusual Time,Low,2026-08-29 18:36:50.122852,True
3,ALT_025868,TX_00025868,CUST_003270,New Device,Low,2026-08-27 17:50:33.578256,True
4,ALT_027966,TX_00027966,CUST_009474,Unusual Time,Medium,2026-09-02 14:15:34.057282,True
...,...,...,...,...,...,...,...
4995,ALT_011481,TX_00011481,CUST_000948,High Amount,High,2026-08-04 05:09:30.281754,False
4996,ALT_086059,TX_00086059,CUST_008245,High Amount,Medium,2026-08-08 11:59:47.914110,False
4997,ALT_029598,TX_00029598,CUST_002021,High Amount,Medium,2026-08-24 18:16:34.423839,True
4998,ALT_077222,TX_00077222,CUST_007000,New Device,Critical,2026-08-20 03:40:45.821913,True
